# Hyper-optimisation : Adult Income (UCI)

Dans cette partie, on va effectuer une hyperoptimisation des paramètres sur tous les modèles optimisables (donc Dummy et Stacking non compris) avec **Optuna**.
On choisit ensuite le modèle final selon son AP (ou selon les autres métriques si égalité) par validation croisée.
Enfin, on choisit le seuil de décision optimal (selon le F1 score car l'AP ne dépend pas du seuil) pour ce modèle en utilisant également une validation croisée

Encore une fois, l'optimisation est ici faite uniquement sur le train set et le test set n'est touché à aucun moment pour éviter la fuite de données


In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from adult_income import (
    DataLoader,
    HyperparameterTuner,
    ModelEvaluator,
    ModelRegistry,
    PreprocessingPipeline,
)

STORAGE = ROOT / "artifacts" / "optuna"
print("Imports OK")

## 1. Données et prétraitement

In [ ]:
loader = DataLoader()
X_train, y_train = loader.load_train()

pp = PreprocessingPipeline(capital_treatment="naive")
y_train_enc = pp.fit_transform_target(y_train)

print(f"Train : {X_train.shape[0]:,} obs, {X_train.shape[1]} variables")
print(f"Prévalence >50K : {y_train_enc.mean():.1%}")

## 2. Hyper-optimisation Optuna

Chaque modèle dispose d'un espace de recherche est optimisé par une étude Optuna
(TPE + pruning médian) qui maximise l'Average Precision (`average_precision`, aire sous la courbe précision-rappel) en CV 5 fold sur le
train. L'Average Precision est privilégiée à la ROC-AUC car elle reflète mieux la performance sur la
classe minoritaire (`>50K`, ~24 %) (comme mentionné précédemmnet)

Les résultats (50 trials/modèle) sont sauvegardés dans `artifacts/optuna/best_params.json`

In [ ]:
RUN_TUNING = True  # True -> relance Optuna + régénère la base SQLite si pas déjà existante (si False, utilise les parametres existants)

TUNABLE = HyperparameterTuner.tunable_models()
print(f"Modèles optimisables ({len(TUNABLE)}) : {TUNABLE}") # stacking et dummy ne sont pas conservés

if RUN_TUNING:
    tuner = HyperparameterTuner(
        n_trials=50, cv=5, scoring="average_precision", n_jobs=4, storage=STORAGE
    )
    for name in TUNABLE:
        print(f"Tuning {name}...")
        tuner.tune(name, pp.build_pipeline(ModelRegistry.get(name)), X_train, y_train_enc)

best = HyperparameterTuner.load_best_params(STORAGE)
HyperparameterTuner.summary_table(best).round(4)

In [ ]:
# Reconstruction des pipelines avec les meilleurs hyperparamètres trouvés
tuned_pipelines = {
    n: HyperparameterTuner.build_tuned_pipeline(pp, n, info["best_params"])
    for n, info in best.items()
}
print("Pipelines optimisés :", list(tuned_pipelines))

## 3. Visualisations Optuna (optionnel)

Historique d'optimisation et importance des hyperparamètres. Ces graphes nécessitent labase SQLite des études (générée seulement si `RUN_TUNING = True`)

In [ ]:
try:
    viz = HyperparameterTuner(storage=STORAGE)
    saved = viz.list_saved_studies()
    if any(s.startswith("xgboost") for s in saved):
        viz.load_study("xgboost")
        display(viz.plot_optimization_history("xgboost"))
        display(viz.plot_param_importances("xgboost"))
    else:
        print("Études Optuna absentes du disque : relancer avec RUN_TUNING=True.")
except Exception as e:
    print("Visualisations indisponibles :", e)

## 4. Sélection multicritère du modèle final

On compare les modèles optimisés en validation croisée sur le train

L'**Average Precision** est prioritaire, et les autres métriques départagent les modèles proches. On
retient le meilleur représentant pour la suite (évaluation finale)

In [ ]:
selector = ModelEvaluator(cv=5, random_state=42)
selector.cross_validate_all(tuned_pipelines, X_train, y_train_enc)

selection_table = selector.cv_comparison_table(sort_by="average_precision")
selection_table[[
    "model", "average_precision_mean", "f1_mean", "roc_auc_mean",
    "precision_mean", "recall_mean", "accuracy_mean",
]].round(4)

### Stabilité du classement : défaut vs optimisé

Cette partie sert simplement à montrer l'écart de classement entre les modèles paramétrés par défaut, et ceux avec leurs paramètres hyperoptimisés. 
On voit que le classement est modifié, ce qui nous fait dire qu'on a bien fait de ne pas prendre de selection particulière de finalistes avant l'hyper optimisation (par exemple les arbres de décision sont beaucoup plus performants une fois hyperopti, alors que la regression logistique et le svm ne bougent quasiment pas)

In [ ]:
default_pipelines = {n: pp.build_pipeline(ModelRegistry.get(n)) for n in best}
default_eval = ModelEvaluator(cv=5, random_state=42)
default_eval.cross_validate_all(default_pipelines, X_train, y_train_enc)

ranking = (
    default_eval.cv_comparison_table()[["model", "average_precision_mean"]]
    .rename(columns={"average_precision_mean": "AP_défaut"})
    .merge(
        selection_table[["model", "average_precision_mean"]]
        .rename(columns={"average_precision_mean": "AP_tuné"}),
        on="model",
    )
)
ranking["Δ"] = (ranking["AP_tuné"] - ranking["AP_défaut"]).round(4)
ranking["rang_défaut"] = ranking["AP_défaut"].rank(ascending=False).astype(int)
ranking["rang_tuné"] = ranking["AP_tuné"].rank(ascending=False).astype(int)
ranking.sort_values("AP_tuné", ascending=False).reset_index(drop=True).round(4)

In [ ]:
best_model_name = selection_table.iloc[0]["model"]
final_params = best[best_model_name]["best_params"]
final_pipeline = HyperparameterTuner.build_tuned_pipeline(pp, best_model_name, final_params)
final_ap = float(selection_table.iloc[0]["average_precision_mean"])

print(f"Modèle final retenu : {best_model_name}")
print(f"Average Precision (CV) : {final_ap:.4f}")
print(f"Hyperparamètres     : {final_params}")

## 5. Courbe d'apprentissage

Évolution de l'Average Precision (sur les données d'entraînement vs sur le fold de validation de la validation croisée) selon la taille du jeu
d'entraînement. Un écart qui se resserre indique une bonne généralisation. Par ailleurs, le plateau qui se forme indique le fait qu'ajouter des données ne suffirait plus à faire augmenter la performance du modèle


In [ ]:
selector.plot_learning_curve(
    final_pipeline, X_train, y_train_enc,
    scoring="average_precision", model_name=best_model_name,
)
plt.show()

## 6. Choix du seuil de décision

Le seuil par défaut (0,5) n'est pas forcément optimal lorsqu'on a un déséquilibre de classes. On le règle via des prédictions sur une cross validation. Une fois les prédictions faites, on recolle le tout et on obtient donc une proba par ligne du train (chacune produite via une validation croisée donc pas de fuite). Ensuite, par le biais d'une table de seuils (81 différents entre 0.10 et 0.90), on compare la proba obtenue avec le seuil pour chaque ligne pour chaque seuil, et on donne une décision, ce qui nous permet de calculer des métriques (F1 score notamment). Ensuite, on choisit le seuil qui maximise le F1-score. Ce seuil sera appliqué tel quel sur le test dans `evaluation.ipynb`.

In [ ]:
y_proba_oof = selector.out_of_fold_probabilities(final_pipeline, X_train, y_train_enc)
best_threshold, threshold_tbl = selector.find_best_threshold(y_train_enc, y_proba_oof, metric="f1")

print(f"Seuil optimal (F1, hors-pli) : {best_threshold:.3f}")
threshold_tbl.sort_values("f1", ascending=False).head(1).round(4)

In [ ]:
selector.plot_threshold_curves(threshold_tbl, best_threshold)
plt.show()

## 7. Sauvegarde pour l'évaluation finale

On enregistre le modèle retenu, ses hyperparamètres et le seuil trouvé dans
`artifacts/final_model.json`, afin de pouvoir l'utiliser dans le notebook d'évaluation


In [ ]:
final_spec = {
    "model_name": best_model_name,
    "best_params": final_params,
    "threshold": best_threshold,
    "scoring": "average_precision",
    "cv_average_precision": final_ap,
}
out_path = ROOT / "artifacts" / "final_model.json"
out_path.write_text(json.dumps(final_spec, indent=2))
print("Sauvegardé ->", out_path)
final_spec